# Module 1 — Take-Home Exercises: Strategy & Data Selection

These exercises reinforce what you learned in class about dataset selection,
data quality auditing, and the go/no-go decision for fine-tuning.

**No GPU required** for Exercises 1-3. Exercise 4 needs a T4 GPU (Colab free tier).

---

## Exercise 1: Evaluate a New Dataset (Conceptual)

You're building a **mental health chatbot** for a university counseling center.
A colleague suggests fine-tuning on the `Amod/mental_health_counseling_conversations`
dataset from Hugging Face.

**Tasks:**

1. Load the dataset and inspect 10 random examples
2. Check for the same quality issues we found in ChatDoctor:
   - Persona contamination (therapist names, platform references)
   - Boilerplate greetings/sign-offs
   - Safety disclaimers (crisis hotline numbers, "seek professional help")
   - Very short responses (<50 characters)
3. Write a 3-sentence assessment: would you fine-tune on this dataset? Why or why not?

In [1]:
!pip install -q datasets pandas

In [2]:
from datasets import load_dataset
import re
import random

# TODO: Load the mental health dataset
ds = load_dataset("Amod/mental_health_counseling_conversations", split="train")

# TODO: Print dataset size and column names
print(f"Dataset size: {len(ds):,} examples")
print(ds.column_names)

# TODO: Display 10 random examples
random.seed(91)
sample_indices = random.sample(range(len(ds)), 10)

for i, idx in enumerate(sample_indices, start=1):
    example = ds[idx]
    print(f"\n{'=' * 80}")
    print(f"Example {i} (index {idx})")
    print(f"{'=' * 80}")
    print(f"\n[Context]\n{example['Context']}")
    print(f"\n[Response]\n{example['Response']}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

combined_dataset.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/3512 [00:00<?, ? examples/s]

Dataset size: 3,512 examples
['Context', 'Response']

Example 1 (index 342)

[Context]
Sometimes I can't stop thinking about life after death. I was raised in a religion that teaches that we will live on forever either in hell or in heaven.  When I think of living forever (even if it is in heaven which should be good), I feel overwhelmed. I don't like the thought of living forever and ever and ever. Sometimes I just can't get the thought out of my mind and the thoughts lead to panic and anxiety.  Am I crazy? I don't think these thoughts are normal.

[Response]
Hi California, What you're experiencing is indeed anxiety; it's very common. Through research we know a lot of things about how to reduce anxiety. I'll get you started and it's a great idea to connect with a therapist who can build on these ideas and get to know you well.It's natural to have random thoughts that unsettle us. Our brains are complex, wonderful things. Fearful and anxious thoughts are a part of this picture; their p

In [3]:
# TODO: Audit for quality issues
# Audit for quality issues
total = len(ds)
# Adapt the audit code from the in-class exercise to check:
#   1. Persona contamination patterns (look for therapist names, platform refs)
persona_patterns = [r"as a therapist", r"as a counselor", r"in my practice",
                    r"my client", r"my patient"]
persona_count = sum(1 for ex in ds if any(re.search(p, ex["Response"].lower()) for p in persona_patterns))
#   2. Boilerplate greetings/sign-offs
boilerplate_patterns = [r"thank you for sharing", r"i appreciate you",
                        r"it takes courage", r"i hear you"]
boilerplate_count = sum(1 for ex in ds if any(re.search(p, ex["Response"].lower()) for p in boilerplate_patterns))

#   3. Safety disclaimers (crisis hotline, "seek professional help", "call 988")
safety_patterns = [r"professional help", r"therapist", r"counselor",
                   r"crisis", r"hotline", r"988", r"emergency"]
safety_count = sum(1 for ex in ds if any(re.search(p, ex["Response"].lower()) for p in safety_patterns))
#   4. Very short responses (<50 chars)
short_count = sum(1 for ex in ds if len(ex["Response"]) < 50)

print(f"Persona contamination: {persona_count:,} ({persona_count/total*100:.1f}%)")
print(f"Boilerplate greetings/sign-offs: {boilerplate_count:,} ({boilerplate_count/total*100:.1f}%)")
print(f"Safety disclaimers: {safety_count:,} ({safety_count/total*100:.1f}%)")
print(f"Very short responses: {short_count:,} ({short_count/total*100:.1f}%)")
# HINT: First read 10-20 examples manually to identify what patterns exist,
#       then write regex patterns to quantify them.

Persona contamination: 115 (3.3%)
Boilerplate greetings/sign-offs: 52 (1.5%)
Safety disclaimers: 1,489 (42.4%)
Very short responses: 9 (0.3%)


**Your Assessment (fill in):**

1. Dataset quality: (good / mixed / poor)
2. Key issue found: ...
3. Would you fine-tune on this? Why or why not?

---

## Exercise 2: Dataset Size vs Quality Trade-off

In class, we learned that 1,000 high-quality examples can beat 10,000 noisy ones.

Given the ChatDoctor dataset (112K examples, 63.1% persona contamination),
answer these questions:

**Tasks:**

1. If you filter out ALL examples with persona contamination, how many remain?
2. If you also filter out examples with boilerplate sign-offs (28.2%), and assume
   some overlap with persona contamination, estimate the remaining clean examples.
3. Would this filtered dataset be better for fine-tuning than the full 112K? Why?
4. Write a filtering function that removes contaminated examples. Run it and
   report the actual count.

In [4]:
from datasets import load_dataset
import re

ds = load_dataset("lavita/ChatDoctor-HealthCareMagic-100k", split="train")
print(f"Original size: {len(ds):,}")

# Persona contamination patterns (from the in-class exercise)
PERSONA_PATTERNS = [
    r"chat\s*doctor", r"welcome to", r"thanks for",
    r"posting (?:your|the) query", r"i understand your concern"
]

BOILERPLATE_PATTERNS = [
    r"hope this helps", r"wishing you", r"good luck",
    r"wish you good health", r"take care"
]

# TODO: Write a function that returns True if an example is CLEAN
# (no persona contamination AND no boilerplate)
def is_clean(example):
    output = example["output"].lower()
    has_persona = any(re.search(p, output) for p in PERSONA_PATTERNS)
    has_boilerplate = any(re.search(p, output) for p in BOILERPLATE_PATTERNS)
    return not has_persona and not has_boilerplate

# TODO: Filter the dataset
clean_ds = ds.filter(is_clean)
print(f"Clean examples: {len(clean_ds):,} ({len(clean_ds)/len(ds)*100:.1f}%)")

README.md:   0%|          | 0.00/542 [00:00<?, ?B/s]

data/train-00000-of-00001-5e7cb295b9cff0(…):   0%|          | 0.00/70.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/112165 [00:00<?, ? examples/s]

Original size: 112,165


Filter:   0%|          | 0/112165 [00:00<?, ? examples/s]

Clean examples: 23,812 (21.2%)


**Your Answers (fill in):**

1. Examples remaining after removing persona contamination: ___
2. Estimated clean examples (no persona, no boilerplate): ___
3. Better for fine-tuning? (yes/no and why): ...
4. Actual count from your filter: ___

---

## Exercise 3: Design Your Own Reformatting Prompt

In class, we used `data_prep_v2.py` to reformat WikiDoc via GPT-4o-mini.
The prompt told GPT-4o-mini to convert terse encyclopedia text into
conversational healthcare-assistant responses.

**Scenario:** You want to fine-tune a model for a **pediatrics helpline**
that helps parents understand common childhood illnesses.

**Tasks:**

1. Write a system prompt for GPT-4o-mini that would reformat medical text
   into parent-friendly answers. Consider:
   - Tone (reassuring but not dismissive)
   - Reading level (non-medical parents)
   - When to say "call your pediatrician" vs "go to the ER now"
   - Safety disclaimers appropriate for worried parents
2. Test your prompt on the sample input below by calling the OpenAI API
   (or write what you think the output should look like)

In [5]:

from google.colab import userdata


# Sample raw medical text (like WikiDoc)
SAMPLE_INPUT = """
Question: What causes croup in children?
Answer: Croup is most commonly caused by parainfluenza viruses, particularly
types 1 and 3. Other causes include influenza A and B, adenovirus, respiratory
syncytial virus (RSV), and measles. The infection causes swelling of the larynx,
trachea, and bronchi. Peak incidence is between 6 months and 3 years of age,
with most cases occurring in autumn.
"""

# TODO: Write your reformatting prompt
REFORMATTING_PROMPT = """
You are reformatting medical text into parent-friendly answers for a
pediatrics helpline chatbot.

Rules:
- Use a warm, reassuring tone. Parents calling are worried.
- Write at a 6th-grade reading level. Avoid jargon — if you must use
  a medical term, explain it in parentheses.
- Always include WHEN TO CALL THE DOCTOR and WHEN TO GO TO THE ER
  as separate clearly labeled sections.
- End with: "Remember, you know your child best. If something doesn't
  feel right, call your pediatrician — that's what they're there for."
- Keep the answer concise and focused. Do not pad with filler.
- Do NOT diagnose. Frame as "common causes include..." not "your child has..."
"""

# TODO (optional): If you have an OpenAI API key, test it:
from openai import OpenAI
client = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": REFORMATTING_PROMPT},
        {"role": "user", "content": SAMPLE_INPUT}
    ]
)
print(response.choices[0].message.content)

Croup is a condition that can make your child's throat swell, which can cause a harsh, barking cough and sometimes make it hard for them to breathe. Common causes include certain viruses, like the parainfluenza virus, influenza, and others. This is most likely to happen in children between 6 months and 3 years old, especially during the fall.

**WHEN TO CALL THE DOCTOR:**
- If your child has a barking cough and you're worried.
- If they have trouble breathing or make a wheezing sound.
- If they seem very tired, are drooling more than usual, or are having trouble swallowing.

**WHEN TO GO TO THE ER:**
- If your child is struggling to breathe or their skin is turning blue.
- If they are very anxious or scared.
- If they suddenly can't speak or cry normally.

Remember, you know your child best. If something doesn't feel right, call your pediatrician — that's what they're there for.


---

## Exercise 4: System Prompt Ablation Study (T4 GPU required)

In class, we tested 3 system prompts. Now test **2 more** to see how far
prompt engineering can go.

**Tasks:**

1. Design a "safety-first" system prompt that prioritizes disclaimers above all else
2. Design a "structured" system prompt that demands bullet points, headers, and numbered lists
3. Run both on the same 10 benchmark questions
4. Compare: which prompt produces the most helpful output? The safest?

In [6]:
!pip install -q transformers torch bitsandbytes accelerate

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map="auto")

BENCHMARK_PROMPTS = [
    "What are the common symptoms of Type 2 diabetes?",
    "How does hypertension affect the heart over time?",
    "What is the recommended first-line treatment for mild persistent asthma?",
    "Explain the difference between Type 1 and Type 2 diabetes.",
    "What are the common side effects of ibuprofen?",
    "How is pneumonia typically diagnosed?",
    "What lifestyle changes help manage high cholesterol?",
    "What are the early warning signs of a stroke?",
    "How does metformin work for diabetes management?",
    "What vaccinations are recommended for adults over 65?",
]

def generate_response(prompt, system_prompt, model, tokenizer, max_new_tokens=300):
    messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, temperature=0.7, top_p=0.9, do_sample=True)
    return tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 40.2 MB/s eta 0:00:00


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [7]:
# TODO: Design your two system prompts

SAFETY_FIRST_PROMPT = """
You are a healthcare information assistant. Your TOP PRIORITY is safety.

Rules (in order of importance):
1. ALWAYS start with: "This is general health information, not medical advice."
2. ALWAYS end with: "Please consult a healthcare professional for personalized guidance."
3. For ANY symptom that could be an emergency (chest pain, stroke signs, difficulty breathing),
   say "Call 911 or go to the nearest emergency room immediately" BEFORE any other information.
4. Never say "you should take" or "you need" — instead say "your doctor may recommend"
5. Include general health information, but frame everything as possibilities, not diagnoses.
"""

STRUCTURED_PROMPT = """
You are a healthcare information assistant. Format EVERY response with:

## Overview
One-paragraph summary.

## Key Points
- Bullet point 1
- Bullet point 2
- (3-5 bullet points)

## What to Know
Numbered list of important details:
1. First detail
2. Second detail

## When to See a Doctor
Specific warning signs that need medical attention.

Always use markdown headers, bullet points, and numbered lists.
Keep paragraphs to 2-3 sentences maximum.
"""



In [9]:
# Run on 3 test questions first to iterate quickly
test_questions = BENCHMARK_PROMPTS[:3]

for q in test_questions:
    print(f"\n{'='*60}")
    print(f"Q: {q}")
    print(f"\n--- SAFETY-FIRST ---")
    print(generate_response(q, SAFETY_FIRST_PROMPT, model, tokenizer))
    print(f"\n--- STRUCTURED ---")
    print(generate_response(q, STRUCTURED_PROMPT, model, tokenizer))


Q: What are the common symptoms of Type 2 diabetes?

--- SAFETY-FIRST ---
The following are some of the most common symptoms of type 2 diabetes:

- Increased thirst and urination 
- Extreme fatigue  
- Blurry vision 
- Slow-healing cuts or scrapes 
- Frequent infections such as yeast infections in women or bacterial infections in men 
- Tingling, numbness or pain in hands and feet 

If you experience these symptoms or have concerns about your health, it's important to speak with your healthcare provider. They can help determine if you have diabetes or another condition by performing tests such as measuring blood sugar levels, checking for nerve damage, and looking for signs of eye or kidney damage. Early diagnosis and treatment are crucial for managing this chronic condition.

--- STRUCTURED ---
Common symptoms of Type 2 diabetes include increased thirst, frequent urination, unexplained weight loss, fatigue, blurred vision, and recurrent infections.

Q: How does hypertension affect th

In [10]:
# TODO: After reviewing the 3-question test, run the full 10-question benchmark
# with your best prompt. Save results for comparison.

results = []
for q in BENCHMARK_PROMPTS:
    results.append({
        "question": q,
        "safety_first": generate_response(q, SAFETY_FIRST_PROMPT, model, tokenizer),
        # "structured": generate_response(q, STRUCTURED_PROMPT, model, tokenizer),
    })

In [14]:
results[0]['safety_first']

"The following are some common symptoms of Type 2 Diabetes:\n\n- Frequent urination \n- Increased thirst and hunger\n- Fatigue  \n- Blurry vision\n- Slow healing wounds\n- Frequent infections like gums and skin infections\n- Tingling or numbness in hands and feet\n- Tiredness after eating\n- Weight loss despite increased appetite\n- Deepened breath odor\n- Dry mouth\n- Unusual sweetening of urine\n- Itching around genital area\n\nIf you experience these symptoms, it's important to see a healthcare provider who can help determine if this might be a sign of diabetes. Early diagnosis and treatment are crucial for managing and controlling the condition."

**Your Findings (fill in):**

1. Which prompt produced more helpful answers? Why?
2. Which prompt produced safer answers? Why?
3. Could any of these prompts match what fine-tuning achieved in Module 2?
4. What does this tell you about when to use prompt engineering vs fine-tuning?